In [1]:
import kagglehub
import os
# Download the "NYC Yellow Taxi Trip Data" variant to a specific directory 
# on your shared cluster filesystem (e.g., your home or scratch directory)
shared_dir = os.path.expanduser("/data/jovillalobos/kaggle")

path = kagglehub.dataset_download(
    "threnjen/2019-airline-delays-and-cancellations",
    output_dir=shared_dir
)

print(f"Dataset downloaded to shared location: {path}")
# Note: Inside this path, look for the CSV name (e.g., 'yellow_tripdata_2016-01.csv')

Dataset downloaded to shared location: /data/jovillalobos/kaggle


In [2]:
import dask.dataframe as dd

# Define your paths

train_path = os.path.join(path, "train.csv")
test_path = os.path.join(path, "test.csv")

parquet_train = os.path.join(path, "train.parquet")
parquet_test = os.path.join(path, "test.parquet")

if not (os.path.exists(parquet_train) or os.path.exists(parquet_test)): 
    
    print("Converting Train CSV to Parquet...")
    # Dask reads the CSV in chunks and streams it to compressed Parquet format
    df_train = dd.read_csv(train_path, assume_missing=True)
    df_train.to_parquet(parquet_train, engine='pyarrow', compression='snappy')
    
    print("Converting Test CSV to Parquet...")
    df_test = dd.read_csv(test_path, assume_missing=True)
    df_test.to_parquet(parquet_test, engine='pyarrow', compression='snappy')
    
    print("Conversion Complete!")

else: 
    print("Already converted!")

Already converted!


In [3]:
def remote_objective(trial, train_path, test_path):
    import gc
    import rmm
    import polars as pl
    import cudf
    from cuml.ensemble import RandomForestClassifier
    from cuml.metrics import accuracy_score

    # 1. Reset the GPU Memory Pool
    if rmm.is_initialized():
        rmm.reinitialize(managed_memory=True)
    else:
        rmm.initialize(managed_memory=True)

    # ==========================================
    # 2. CPU Data Processing (Polars)
    # ==========================================
    target_col = "DEP_DEL15"
    cat_cols = [
        "DEP_TIME_BLK",
        "CARRIER_NAME",
        "DEPARTING_AIRPORT",
        "PREVIOUS_AIRPORT",
    ]

    # Load data
    train_pl = pl.read_parquet(train_path).drop_nulls()
    test_pl = pl.read_parquet(test_path).drop_nulls()

    # Consistent categorical encoding
    for col in cat_cols:
        if col in train_pl.columns:
            unique_values = (
                pl.concat([
                    train_pl.select(col),
                    test_pl.select(col)
                ])
                .unique()
                .with_row_index("code")
            )

            train_pl = (
                train_pl
                .join(unique_values, on=col)
                .drop(col)
                .rename({"code": col})
            )

            test_pl = (
                test_pl
                .join(unique_values, on=col)
                .drop(col)
                .rename({"code": col})
            )

    features = [c for c in train_pl.columns if c != target_col]

    # Cast to float32/int32 during Pandas conversion for GPU efficiency
    X_train_pd = train_pl.select(features).to_pandas().astype('float32')
    y_train_pd = train_pl[target_col].to_pandas().astype('int32')
    X_test_pd = test_pl.select(features).to_pandas().astype('float32')
    y_test_pd = test_pl[target_col].to_pandas().astype('int32')

    # Aggressive CPU cleanup
    del train_pl, test_pl
    gc.collect()

    # ==========================================
    # 3. GPU Handoff & Training (RAPIDS)
    # ==========================================
    
    # Load into L40S VRAM
    X_train = cudf.DataFrame.from_pandas(X_train_pd)
    y_train = cudf.Series.from_pandas(y_train_pd)
    X_test = cudf.DataFrame.from_pandas(X_test_pd)
    y_test = cudf.Series.from_pandas(y_test_pd)

    # Free up host node memory
    del X_train_pd, y_train_pd, X_test_pd, y_test_pd
    gc.collect()

    params = {
        "n_estimators": trial.suggest_int("n_estimators", 50, 300, step=25),
        "max_depth": trial.suggest_int("max_depth", 6, 16),
        "max_features": trial.suggest_float("max_features", 0.5, 0.9),
        "n_streams": 8
    }
    
    # Train Model
    model = RandomForestClassifier(
        **params,
        n_bins=128,
        random_state=42
    )
    model.fit(X_train, y_train)

    # Predict
    preds = model.predict(X_test)
    acc = float(accuracy_score(y_test, preds))

    # Final sweep of GPU memory before returning to Optuna
    del X_train, y_train, X_test, y_test, model, preds
    gc.collect()

    return acc

In [ ]:
# Because these are strings, Dask will serialize this partial perfectly!
import optuna 
from functools import partial

objective_partial = partial(
    remote_objective,
    train_path=parquet_train,
    test_path=parquet_test
)

study = optuna.create_study(
    direction="maximize",
)

print("Submitting distributed study.optimize trials...")

# 5. Execute the search loop
study.optimize(objective_partial, n_trials=15)
               
print("\n--- Optimization Complete ---")
print(f"Best Accuracy: {study.best_value:.4f}")
print(f"Best Configuration: {study.best_params}")

[I 2026-06-10 14:56:02,872] A new study created in memory with name: no-name-c03da257-166b-44d6-a34e-27bba40ef39f


Submitting distributed study.optimize trials...


[I 2026-06-10 14:56:47,976] Trial 0 finished with value: 0.823048421472231 and parameters: {'n_estimators': 75, 'max_depth': 15, 'max_features': 0.892929061911101}. Best is trial 0 with value: 0.823048421472231.
[I 2026-06-10 14:57:42,650] Trial 1 finished with value: 0.8204142457129149 and parameters: {'n_estimators': 125, 'max_depth': 13, 'max_features': 0.6827155258914562}. Best is trial 0 with value: 0.823048421472231.
[I 2026-06-10 14:59:09,306] Trial 2 finished with value: 0.8191772926652485 and parameters: {'n_estimators': 225, 'max_depth': 12, 'max_features': 0.6351915482323717}. Best is trial 0 with value: 0.823048421472231.
[I 2026-06-10 15:01:17,666] Trial 3 finished with value: 0.8206258838589442 and parameters: {'n_estimators': 300, 'max_depth': 13, 'max_features': 0.7032786174161241}. Best is trial 0 with value: 0.823048421472231.
[I 2026-06-10 15:02:04,535] Trial 4 finished with value: 0.8214744911823432 and parameters: {'n_estimators': 100, 'max_depth': 14, 'max_feature